# モデルの保存と読み込み

---
## 目的
学習したネットワークのパラメータ（`state_dict`）を保存・読み込みする方法を理解する．
また，モデルのパラメータだけでなくoptimizerの状態やエポック数などもあわせて保存し，学習を途中から再開する方法を身につける．

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
from time import time

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchsummary

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットとネットワークの準備
MNISTデータセットと畳み込みニューラルネットワーク（CNN）を用意します．

In [ ]:
train_data = torchvision.datasets.MNIST(root="./", train=True, transform=transforms.ToTensor(), download=True)
test_data = torchvision.datasets.MNIST(root="./", train=False, transform=transforms.ToTensor(), download=True)

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 4, kernel_size=3, stride=1, padding=1)
        self.l1 = nn.Linear(14*14*4, 10)
        self.act = nn.Sigmoid()
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        h = self.pool(self.act(self.conv1(x)))
        h = h.view(h.size()[0], -1)
        h = self.l1(h)
        return h

In [ ]:
model = CNN()
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
criterion = nn.CrossEntropyLoss()
criterion = criterion.to(device)

torchsummary.summary(model, (1, 28, 28), device=device.type)

## パラメータ（state_dict）の保存と読み込み

PyTorchのネットワークモデルは，`state_dict()`というメソッドにより，各層の名前とパラメータ（重み・バイアス）を対応付けた辞書（`OrderedDict`）としてパラメータを取得できます．
モデルを保存する際は，モデルのオブジェクトそのものではなく，この`state_dict`を保存することが推奨されています．

**モデルのオブジェクトごと保存する方法（`torch.save(model, path)`）もありますが，保存時のクラス定義やファイル構成に依存してしまい，コードを少し変更しただけで読み込めなくなることがあるため推奨されません．**
`state_dict`はパラメータの値のみを保存するため，読み込み時にはあらかじめ同じ構造のネットワーク（`CNN`クラスのインスタンス）を用意しておく必要があります．

保存には`torch.save()`，読み込みには`torch.load()`と`load_state_dict()`を使用します．

`torch.load()`の`map_location`には，パラメータを読み込む際に配置するデバイスを指定します．
GPU上で保存したパラメータをCPUのみの環境で読み込む場合など，保存時と読み込み時でデバイス構成が異なる場合には，この指定を忘れるとエラーになることがあるため注意してください．

In [ ]:
### 学習前のパラメータの一部を表示
print(model.conv1.bias)

### state_dictの保存
torch.save(model.state_dict(), 'model_weights.pth')

### 新しいネットワークを作成し，保存したパラメータを読み込む
loaded_model = CNN()
loaded_model.load_state_dict(torch.load('model_weights.pth', map_location=device))
loaded_model = loaded_model.to(device)

### 読み込んだパラメータが元のモデルと一致しているか確認
print(loaded_model.conv1.bias)
print(torch.equal(model.conv1.bias, loaded_model.conv1.bias))

## チェックポイントの保存

学習を途中から再開するためには，ネットワークのパラメータだけでは不十分です．
`optimizer`はモーメンタムや過去の勾配の情報など，パラメータ更新に関する内部状態（`state_dict`）を保持しており，これを保存しておかないと，学習を再開した直後の更新が不安定になってしまいます．
また，どのエポックまで学習が完了しているかという情報も必要です．

そこで，モデルとoptimizerの`state_dict`に加えて，エポック数や損失の値などをまとめた辞書を作成し，`torch.save()`で保存します．
このように，学習を再開するために必要な情報一式をまとめて保存したものを**チェックポイント（checkpoint）**と呼びます．

In [ ]:
batch_size = 100
epoch_num = 3

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)

model.train()

train_start = time()
for epoch in range(1, epoch_num+1):
    sum_loss = 0.0
    count = 0

    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)
        loss = criterion(y, label)

        model.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

    mean_loss = sum_loss / len(train_loader)
    print("epoch: {}, mean loss: {}, mean accuracy: {}, elapsed time: {}".format(epoch, mean_loss, count.item()/len(train_data), time() - train_start))

    ### チェックポイントの保存（1エポックごとに上書き）
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': mean_loss,
    }
    torch.save(checkpoint, 'checkpoint.pth')

## チェックポイントからの学習再開

保存したチェックポイントを読み込み，ネットワークとoptimizerの状態を復元したうえで，中断したエポックの続きから学習を再開します．
`optimizer`はモデルのパラメータを渡して作成した後に，`load_state_dict()`で保存しておいた内部状態を読み込む点に注意してください．

In [ ]:
### ネットワークとoptimizerを新しく作成
resumed_model = CNN()
resumed_model = resumed_model.to(device)
resumed_optimizer = torch.optim.SGD(resumed_model.parameters(), lr=0.01, momentum=0.9)

### チェックポイントの読み込み
checkpoint = torch.load('checkpoint.pth', map_location=device)
resumed_model.load_state_dict(checkpoint['model_state_dict'])
resumed_optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
start_epoch = checkpoint['epoch'] + 1

print("epoch {} まで学習済み（loss: {:.4f}）．epoch {} から再開します．".format(checkpoint['epoch'], checkpoint['loss'], start_epoch))

### 続きから学習を再開
resumed_model.train()

train_start = time()
for epoch in range(start_epoch, start_epoch + epoch_num):
    sum_loss = 0.0
    count = 0

    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        y = resumed_model(image)
        loss = criterion(y, label)

        resumed_model.zero_grad()
        loss.backward()
        resumed_optimizer.step()

        sum_loss += loss.item()
        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

    print("epoch: {}, mean loss: {}, mean accuracy: {}, elapsed time: {}".format(epoch, sum_loss/len(train_loader), count.item()/len(train_data), time() - train_start))

## テスト
再開した学習によって得られたネットワークを用いて評価を行います．

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=100, shuffle=False)

resumed_model.eval()

count = 0
with torch.no_grad():
    for image, label in test_loader:
        image = image.to(device)
        label = label.to(device)

        y = resumed_model(image)

        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

print("test accuracy: {}".format(count.item() / len(test_data)))

## 課題

1. 一定エポックごとにチェックポイントを保存するのではなく，これまでで最も検証データでの精度（またはlossの値）が良かった時のみ保存する「ベストモデル保存」の仕組みを実装してみましょう．

2. チェックポイントの保存先のファイル名にエポック数を含めて（例：`checkpoint_epoch_0005.pth`），複数のチェックポイントを別々のファイルとして残すように変更してみましょう．